In [ ]:
import cv2
import os
import numpy as np

# ==========================================================
# CONFIGURATION
# ==========================================================

dataset_path = "4228/Pre"
output_path = "4228/processed_faces"

FACE_SIZE = (90, 90)

os.makedirs(output_path, exist_ok=True)

# ==========================================================
# LOAD CASCADE CLASSIFIERS (Viola–Jones via OpenCV)
# ==========================================================

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

eye_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_eye.xml"
)

# ==========================================================
# FACE ALIGNMENT FUNCTION
# ==========================================================

def align_face(gray_image, face_rect):
    """
    Align face by detecting two eyes and rotating so that
    they are horizontally aligned.
    """
    (x, y, w, h) = face_rect
    face_roi = gray_image[y:y+h, x:x+w]

    # Slight blur reduces false eye detections
    face_roi = cv2.GaussianBlur(face_roi, (5, 5), 0)

    eyes = eye_cascade.detectMultiScale(
        face_roi,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(15, 15)
    )

    # Need at least 2 eyes
    if len(eyes) < 2:
        return gray_image

    # Keep only eyes in upper 60% of face
    filtered_eyes = []
    for (ex, ey, ew, eh) in eyes:
        if ey < h * 0.6:
            filtered_eyes.append((ex, ey, ew, eh))

    if len(filtered_eyes) < 2:
        return gray_image

    # Select pair of eyes closest in vertical alignment
    min_diff = float("inf")
    best_pair = None

    for i in range(len(filtered_eyes)):
        for j in range(i + 1, len(filtered_eyes)):
            y1 = filtered_eyes[i][1]
            y2 = filtered_eyes[j][1]
            diff = abs(y1 - y2)

            if diff < min_diff:
                min_diff = diff
                best_pair = (filtered_eyes[i], filtered_eyes[j])

    if best_pair is None:
        return gray_image

    eye1, eye2 = best_pair

    # Sort left to right
    if eye1[0] > eye2[0]:
        eye1, eye2 = eye2, eye1

    # Compute eye centers
    eye1_center = (
        eye1[0] + eye1[2] // 2,
        eye1[1] + eye1[3] // 2
    )

    eye2_center = (
        eye2[0] + eye2[2] // 2,
        eye2[1] + eye2[3] // 2
    )

    dx = eye2_center[0] - eye1_center[0]
    dy = eye2_center[1] - eye1_center[1]

    angle = np.degrees(np.arctan2(dy, dx))
    angle = float(angle)

    # Ignore unrealistic rotation
    if abs(angle) > 20:
        return gray_image

    # Rotate entire image around face center
    cx = float(x) + float(w) / 2.0
    cy = float(y) + float(h) / 2.0
    center = (cx, cy)

    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)

    rotated = cv2.warpAffine(
        gray_image,
        rotation_matrix,
        gray_image.shape[::-1],
        flags=cv2.INTER_CUBIC
    )

    return rotated


# ==========================================================
# MAIN PROCESSING LOOP
# ==========================================================

success_count = 0
fail_count = 0

for person_name in os.listdir(dataset_path):

    person_input_folder = os.path.join(dataset_path, person_name)

    if not os.path.isdir(person_input_folder):
        continue

    person_output_folder = os.path.join(output_path, person_name)
    os.makedirs(person_output_folder, exist_ok=True)

    print(f"\nProcessing: {person_name}")

    for filename in os.listdir(person_input_folder):

        image_path = os.path.join(person_input_folder, filename)
        image = cv2.imread(image_path)

        if image is None:
            print(f"Could not read: {filename}")
            continue

        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # ------------------------------------------------------
        # STEP 1: Face Detection
        # ------------------------------------------------------
        faces = face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(40, 40)
        )

        if len(faces) == 0:
            print(f"No face detected: {filename}")
            fail_count += 1
            continue

        # Choose largest face
        faces = sorted(faces, key=lambda r: r[2] * r[3], reverse=True)
        (x, y, w, h) = faces[0]

        # ------------------------------------------------------
        # STEP 2: Alignment
        # ------------------------------------------------------
        aligned_gray = align_face(gray, (x, y, w, h))

        # Re-detect face after rotation
        faces_aligned = face_cascade.detectMultiScale(
            aligned_gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(40, 40)
        )

        if len(faces_aligned) > 0:
            faces_aligned = sorted(faces_aligned,
                                   key=lambda r: r[2] * r[3],
                                   reverse=True)
            (x, y, w, h) = faces_aligned[0]

        # ------------------------------------------------------
        # STEP 3: Add Margin
        # ------------------------------------------------------
        margin = int(0.2 * max(w, h))

        x = max(0, x - margin)
        y = max(0, y - margin)
        w = min(aligned_gray.shape[1] - x, w + 2 * margin)
        h = min(aligned_gray.shape[0] - y, h + 2 * margin)

        face_crop = aligned_gray[y:y+h, x:x+w]

        if face_crop.size == 0:
            print(f"Invalid crop: {filename}")
            fail_count += 1
            continue

        # ------------------------------------------------------
        # STEP 4: Resize
        # ------------------------------------------------------
        resized = cv2.resize(
            face_crop,
            FACE_SIZE,
            interpolation=cv2.INTER_AREA
        )

        # ------------------------------------------------------
        # STEP 5: CLAHE Normalization
        # ------------------------------------------------------
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        normalized = clahe.apply(resized)

        # ------------------------------------------------------
        # STEP 6: Save
        # ------------------------------------------------------
        save_path = os.path.join(person_output_folder, filename)
        cv2.imwrite(save_path, normalized)

        success_count += 1
        print(f"Processed: {filename}")

# ==========================================================
# SUMMARY
# ==========================================================

print("\n===================================")
print(f"Successfully processed: {success_count}")
print(f"Failed: {fail_count}")
print("Preprocessing complete.")
print("===================================")